In [0]:
# !wget https://developer.nvidia.com/compute/cuda/9.2/Prod/local_installers/cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64 -O cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb
# !dpkg -i cuda-repo-ubuntu1604-9-2-local_9.2.88-1_amd64.deb
# !apt-key add /var/cuda-repo-9-2-local/7fa2af80.pub
# !apt-get update
# !apt-get install cuda

In [0]:
# !pip install -q tensorflow-gpu==1.13.0rc0 opencv-python

In [13]:
import numpy as np
import os
import six.moves.urllib as urllib
import sys
import tarfile
import tensorflow as tf
import zipfile

from distutils.version import StrictVersion
from collections import defaultdict
from io import StringIO
from matplotlib import pyplot as plt
from PIL import Image

if StrictVersion(tf.__version__.split('-')[0]) < StrictVersion('1.9'):
  raise ImportError('Please upgrade your TensorFlow installation to v1.9.* or later!')

import cv2,time
print('Using OpenCV version %r and Tensorflow version %r'%(cv2.__version__,tf.__version__))

Using OpenCV version '3.4.3' and Tensorflow version '1.13.0-rc0'


In [16]:
#env preparation since this notebook is being run as standalone
!rm -rf *
!git clone https://github.com/tensorflow/models.git md --recursive
!git clone https://github.com/cocodataset/cocoapi.git
!cd cocoapi/PythonAPI && make && cp -rv pycocotools ../../md/research/
!cd md/research && protoc object_detection/protos/*.proto --python_out=.
!cd md/research && python setup.py install
!cd md/research/slim && python setup.py install
!cd md/research && python object_detection/builders/model_builder_test.py
!mv -v md/research/* ./
#!mv md/research/object_detection ./
!mv -v md/research/setup.py ./
!rm -rf md
!ls
!python object_detection/builders/model_builder_test.py

Cloning into 'md'...
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 23718 (delta 0), reused 0 (delta 0), pack-reused 23714
Receiving objects: 100% (23718/23718), 505.53 MiB | 26.93 MiB/s, done.
Resolving deltas: 100% (13991/13991), done.
Checking out files: 100% (2768/2768), done.
Submodule 'tensorflow' (https://github.com/tensorflow/tensorflow.git) registered for path 'research/syntaxnet/tensorflow'
Cloning into '/content/md/research/syntaxnet/tensorflow'...
remote: Enumerating objects: 82, done.        
remote: Counting objects: 100% (82/82), done.        
remote: Compressing objects: 100% (64/64), done.        
remote: Total 518208 (delta 19), reused 29 (delta 18), pack-reused 518126        
Receiving objects: 100% (518208/518208), 299.80 MiB | 23.03 MiB/s, done.
Resolving deltas: 100% (416030/416030), done.
Submodule path 'research/syntaxnet/tensorflow': checked out '8753e2ebde6c58b56675

In [0]:
# This is needed since the notebook is stored in the object_detection folder.
sys.path.append("..")
sys.path.append("object_detection")
from object_detection.utils import ops as utils_ops
from object_detection import utils
from utils import label_map_util
from utils import visualization_utils as vis_util

In [18]:
!cp -rv object_detection/* ./

'object_detection/anchor_generators' -> './anchor_generators'
'object_detection/anchor_generators/__init__.py' -> './anchor_generators/__init__.py'
'object_detection/anchor_generators/grid_anchor_generator.py' -> './anchor_generators/grid_anchor_generator.py'
'object_detection/anchor_generators/grid_anchor_generator_test.py' -> './anchor_generators/grid_anchor_generator_test.py'
'object_detection/anchor_generators/multiple_grid_anchor_generator.py' -> './anchor_generators/multiple_grid_anchor_generator.py'
'object_detection/anchor_generators/multiple_grid_anchor_generator_test.py' -> './anchor_generators/multiple_grid_anchor_generator_test.py'
'object_detection/anchor_generators/multiscale_grid_anchor_generator.py' -> './anchor_generators/multiscale_grid_anchor_generator.py'
'object_detection/anchor_generators/multiscale_grid_anchor_generator_test.py' -> './anchor_generators/multiscale_grid_anchor_generator_test.py'
'object_detection/box_coders' -> './box_coders'
'object_detection/box_

In [19]:
!wget -nc https://github.com/manuhg/masknet/raw/master/input_video.mp4
!wget -nc https://github.com/manuhg/masknet/raw/master/input_video_vs.mp4
input_file='input_video_vs.mp4'

--2019-02-08 04:13:39--  https://github.com/manuhg/masknet/raw/master/input_video.mp4
Resolving github.com (github.com)... 192.30.253.113, 192.30.253.112
Connecting to github.com (github.com)|192.30.253.113|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/manuhg/masknet/master/input_video.mp4 [following]
--2019-02-08 04:13:40--  https://raw.githubusercontent.com/manuhg/masknet/master/input_video.mp4
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 151.101.0.133, 151.101.64.133, 151.101.128.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|151.101.0.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 51626864 (49M) [application/octet-stream]
Saving to: ‘input_video.mp4’

input_video.mp4     100%[===================>]  49.23M   175MB/s    in 0.3s    

2019-02-08 04:13:41 (175 MB/s) - ‘input_video.mp4’ saved [51626864/51626864]

--2019-02-08 04:13:42--  ht

In [0]:
%matplotlib inline

In [0]:
# What model to download.
MODEL_NAME = 'ssd_mobilenet_v1_coco_2017_11_17'
MODEL_FILE = MODEL_NAME + '.tar.gz'
DOWNLOAD_BASE = 'http://download.tensorflow.org/models/object_detection/'

# Path to frozen detection graph. This is the actual model that is used for the object detection.
PATH_TO_FROZEN_GRAPH = MODEL_NAME + '/frozen_inference_graph.pb'

# List of the strings that is used to add correct label for each box.
PATH_TO_LABELS = os.path.join('data', 'mscoco_label_map.pbtxt')

In [0]:
opener = urllib.request.URLopener()
opener.retrieve(DOWNLOAD_BASE + MODEL_FILE, MODEL_FILE)
tar_file = tarfile.open(MODEL_FILE)
for file in tar_file.getmembers():
  file_name = os.path.basename(file.name)
  if 'frozen_inference_graph.pb' in file_name:
    tar_file.extract(file, os.getcwd())

In [0]:
detection_graph = tf.Graph()
with detection_graph.as_default():
  od_graph_def = tf.GraphDef()
  with tf.gfile.GFile(PATH_TO_FROZEN_GRAPH, 'rb') as fid:
    serialized_graph = fid.read()
    od_graph_def.ParseFromString(serialized_graph)
    tf.import_graph_def(od_graph_def, name='')

In [0]:
category_index = label_map_util.create_category_index_from_labelmap(PATH_TO_LABELS, use_display_name=True)

In [0]:
def load_image_into_numpy_array(image):
  (im_width, im_height) = image.size
  return np.array(image.getdata()).reshape(
      (im_height, im_width, 3)).astype(np.uint8)

In [0]:
# For the sake of simplicity we will use only 2 images:
# image1.jpg
# image2.jpg
# If you want to test the code with your images, just add path to the images to the TEST_IMAGE_PATHS.
PATH_TO_TEST_IMAGES_DIR = 'test_images'
TEST_IMAGE_PATHS = [ os.path.join(PATH_TO_TEST_IMAGES_DIR, 'image{}.jpg'.format(i)) for i in range(1, 3) ]

# Size, in inches, of the output images.
IMAGE_SIZE = (12, 8)

In [0]:
def run_inference_for_single_image(image, graph):
  with graph.as_default():
    with tf.Session() as sess:
      # Get handles to input and output tensors
      ops = tf.get_default_graph().get_operations()
      all_tensor_names = {output.name for op in ops for output in op.outputs}
      tensor_dict = {}
      for key in [
          'num_detections', 'detection_boxes', 'detection_scores',
          'detection_classes', 'detection_masks'
      ]:
        tensor_name = key + ':0'
        if tensor_name in all_tensor_names:
          tensor_dict[key] = tf.get_default_graph().get_tensor_by_name(
              tensor_name)
      if 'detection_masks' in tensor_dict:
        # The following processing is only for single image
        detection_boxes = tf.squeeze(tensor_dict['detection_boxes'], [0])
        detection_masks = tf.squeeze(tensor_dict['detection_masks'], [0])
        # Reframe is required to translate mask from box coordinates to image coordinates and fit the image size.
        real_num_detection = tf.cast(tensor_dict['num_detections'][0], tf.int32)
        detection_boxes = tf.slice(detection_boxes, [0, 0], [real_num_detection, -1])
        detection_masks = tf.slice(detection_masks, [0, 0, 0], [real_num_detection, -1, -1])
        detection_masks_reframed = utils_ops.reframe_box_masks_to_image_masks(
            detection_masks, detection_boxes, image.shape[0], image.shape[1])
        detection_masks_reframed = tf.cast(
            tf.greater(detection_masks_reframed, 0.5), tf.uint8)
        # Follow the convention by adding back the batch dimension
        tensor_dict['detection_masks'] = tf.expand_dims(
            detection_masks_reframed, 0)
      image_tensor = tf.get_default_graph().get_tensor_by_name('image_tensor:0')

      # Run inference
      output_dict = sess.run(tensor_dict,
                             feed_dict={image_tensor: np.expand_dims(image, 0)})

      # all outputs are float32 numpy arrays, so convert types as appropriate
      output_dict['num_detections'] = int(output_dict['num_detections'][0])
      output_dict['detection_classes'] = output_dict[
          'detection_classes'][0].astype(np.uint8)
      output_dict['detection_boxes'] = output_dict['detection_boxes'][0]
      output_dict['detection_scores'] = output_dict['detection_scores'][0]
      if 'detection_masks' in output_dict:
        output_dict['detection_masks'] = output_dict['detection_masks'][0]
  return output_dict

In [0]:
import cv2
def detector(image,class_labels,opfile):
  image_np = image
  # Expand dimensions since the model expects images to have shape: [1, None, None, 3]
  image_np_expanded = np.expand_dims(image_np, axis=0)
  # Actual detection.
  output_dict = run_inference_for_single_image(image_np, detection_graph)
  # Visualization of the results of a detection.
  class_labels_detected = [ category_index[obj]['name'] for obj in output_dict['detection_classes'] ]
  print(set(class_labels_detected)&set(class_labels))
  vis_util.visualize_boxes_and_labels_on_image_array(image_np,output_dict['detection_boxes'],output_dict['detection_classes'],output_dict['detection_scores'],
                                              category_index,instance_masks=output_dict.get('detection_masks'),use_normalized_coordinates=True,line_thickness=8)  
  plt.figure(figsize=IMAGE_SIZE)
  plt.imshow(image_np)
  cv2.imwrite(opfile,image_np)

In [0]:
# for image_path in TEST_IMAGE_PATHS:
#   image = Image.open(image_path)  
#   # the array based representation of the image will be used later in order to prepare the
#   # result image with boxes and labels on it.
#   image_np = load_image_into_numpy_array(image)
#   # Expand dimensions since the model expects images to have shape: [1, None, None, 3]
#   image_np_expanded = np.expand_dims(image_np, axis=0)
#   # Actual detection.
#   output_dict = run_inference_for_single_image(image_np, detection_graph)
#   # Visualization of the results of a detection.
#   vis_util.visualize_boxes_and_labels_on_image_array(
#       image_np,
#       output_dict['detection_boxes'],
#       output_dict['detection_classes'],
#       output_dict['detection_scores'],
#       category_index,
#       instance_masks=output_dict.get('detection_masks'),
#       use_normalized_coordinates=True,
#       line_thickness=8)
#   plt.figure(figsize=IMAGE_SIZE)
#   plt.imshow(image_np)

In [0]:
import time
import cv2

def extract_frames(input_file,class_labels,dest_dir='.',interval=None): #interval if specified should be in terms of seconds
  interval = interval * 1000 # convert to milliseconds
  cap = cv2.VideoCapture(input_file)
  labels_matched = {} #format: File name : [list of matched labels]
  if (cap.isOpened()== False): 
    print("Error opening video file",input_file)
  i,count = 0,0
  fps = cap.get(cv2.CAP_PROP_FPS)
  input_file_name =  '.'.join(input_file.split('.')[:-1])
  
  t1 = time.time()
  if dest_dir[-1] != '/':
      dest_dir = dest_dir + '/'

  while(cap.isOpened()):
    if interval:
      cap.set(cv2.CAP_PROP_POS_MSEC,interval)
    ret, frame = cap.read()
    frame = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    i+=1
    if ret == True:
      opfname = input_file_name+'-'+str(int(i/fps))+':'+str(i%fps)+'.jpg'
      class_labels_matched = detector(frame,class_labels,dest_dir+opfname)
      count+=1
      if class_labels_matched:
        labels_matched.update({opfname:class_labels_matched})
    else:
      break
  t2 = time.time()
  print('Overall Processing speed:',(t2-t1)/i)
  print('Frames with detections:{0}/{1}'%(count,i))
  cap.release()
  cv2.destroyAllWindows()
  return labels_matched

In [0]:
extract_frames(input_file,['person'],'.',2)

{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
{'person'}
